In [1]:
import pandas as pd
import json

# Load cleaned data
df = pd.read_csv("bird_migration_clean.csv")

# Save for frontend use
df.to_json("bird_migration_clean.json", orient="records")

# Create GeoJSON routes
features = []
for _, row in df.iterrows():
    features.append({
        "type": "Feature",
        "properties": {
            "Bird_ID": row['Bird_ID'],
            "Species": row['Species'],
            "Risk_Score": row['Risk_Score'],
            "Migration_Success": row['Migration_Success'],
            "Region": row['Region'],
            "Interrupted_Reason": row.get('Interrupted_Reason', "")
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [row['Start_Longitude'], row['Start_Latitude']],
                [row['End_Longitude'], row['End_Latitude']]
            ]
        }
    })
with open("routes.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)

# Precompute main dashboard stats
dashboard_stats = {
    "birds_shown": int(df['Bird_ID'].nunique()),
    "high_risk_segments": int((df['Risk_Score'] >= 3).sum()),
    "distinct_species": int(df['Species'].nunique()),
    "migration_success_rate": f"{(df['Migration_Success']=='Yes').mean():.1%}"
}
with open("dashboard_stats.json", "w") as f:
    json.dump(dashboard_stats, f)

# Scenario predictions (example: two wind scenarios)
from sklearn.ensemble import RandomForestClassifier
features = ['Min_Altitude_m', 'Wind_Speed_kmph', 'Visibility_km', 'Predator_Sightings']
X = df[features]
y = (df['Risk_Score'] >= 3).astype(int)
model = RandomForestClassifier(n_estimators=30, random_state=42)
model.fit(X, y)

scenarios = []
for wind in ["Normal", "High Winds"]:
    wind_speed = df['Wind_Speed_kmph'].mean() + (10 if wind == 'High Winds' else 0)
    test_point = [[120, wind_speed, 8, 1]]
    risk = model.predict(test_point)[0]
    scenarios.append({
        "infra": 0,
        "wind": wind,
        "risk": int(risk),
        "risk_label": "🟥 High Risk" if risk else "🟩 Low Risk"
    })
with open("scenarios.json", "w") as f:
    json.dump(scenarios, f)

# Key narrative/insights
avg_risk = df['Risk_Score'].mean()
top_species = df.groupby('Species')['Risk_Score'].mean().sort_values(ascending=False).head(1).index[0]
top_region = df['Region'].mode()[0]
top_disruption = df['Interrupted_Reason'].mode()[0] if 'Interrupted_Reason' in df else "None"
insights = [
    f"Average risk score: {avg_risk:.2f}",
    f"Highest risk species: {top_species}",
    f"Most common migration region: {top_region}",
    f"Top reason for interruption: {top_disruption}"
]
with open("insights.json", "w") as f:
    json.dump(insights, f)


C:\Users\farzpat.ANT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\farzpat.ANT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
